In [41]:
import pandas as pd
import requests
import numpy as np
import re
import datetime as dt

# Acesso à Base de Dados
Neste notebook serão codificadas duas formas de acesso de dados, uma via requisição HTTP (busca no servidor) e a outra via arquivo direto guardado localmente na memória.

In [6]:
# Acessando a base de dados via requisição HTTP:
# headers = {'User-Agent': 'Chrome/120.0.0.0'}
# response = requests.get('https://cdn3.gnarususercontent.com.br/2928-transformacao-manipulacao-dados/dados_vendas_clientes.json', headers=headers)
# data_frame = pd.read_json(response.text)

In [23]:
# Acessando via arquivo da base de dados armazenado localmente:
data_frame = pd.read_json('/home/felipe-rrgama/Área de trabalho/customer_highest_purchase_analisys/dataframe_archive/dados_vendas_clientes.json')

In [24]:
# Visualizando o dataframe:
data_frame.head(10)

,dados_vendas
0,"{'Data de venda': '06/06/2022', 'Cliente': ['@..."
1,"{'Data de venda': '07/06/2022', 'Cliente': ['I..."
2,"{'Data de venda': '08/06/2022', 'Cliente': ['I..."
3,"{'Data de venda': '09/06/2022', 'Cliente': ['J..."
4,"{'Data de venda': '10/06/2022', 'Cliente': ['M..."


In [25]:
# Como os dados estão aninhados, vamos normalizá-los:
data_frame = pd.json_normalize(data_frame['dados_vendas'])

In [26]:
data_frame = data_frame.explode(list(data_frame.columns[1:3]), ignore_index=True)

data_frame.head()

,Data de venda,Cliente,Valor da compra
0,06/06/2022,@ANA _LUCIA 321,"R$ 836,5"
1,06/06/2022,DieGO ARMANDIU 210,"R$ 573,33"
2,06/06/2022,DieGO ARMANDIU 210,"R$ 392,8"
3,06/06/2022,DieGO ARMANDIU 210,"R$ 512,34"
4,07/06/2022,Isabely JOanes 738,"R$ 825,31"


In [27]:
# Verificando os tipos de dados de cada coluna da tabela:
data_frame.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Data de venda    20 non-null     str  
 1   Cliente          20 non-null     str  
 2   Valor da compra  20 non-null     str  
dtypes: str(3)
memory usage: 612.0 bytes


In [28]:
# A coluna Valor da compra deve ser uma coluna numérica, logo, façamos o processo de transformação do tipo str para o tipo float32
data_frame['Valor da compra'] = data_frame['Valor da compra'].apply(lambda x: x.replace('R$ ', '').replace(',', '.').strip())

data_frame['Valor da compra'] = data_frame['Valor da compra'].astype(np.float32)

data_frame['Valor da compra'] = data_frame['Valor da compra'].apply(lambda x: round(x, 2))

data_frame.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Data de venda    20 non-null     str    
 1   Cliente          20 non-null     str    
 2   Valor da compra  20 non-null     float64
dtypes: float64(1), str(2)
memory usage: 612.0 bytes


In [49]:
# Os valores da coluna 'Cliente' estão em um formato incomun para nomes de pessoas. Portanto, realizemos o processo de tratamento desses nomes, deixando-os em minúsculas e depois removendo os caracteres incomuns:
data_frame['Cliente'] = data_frame['Cliente'].str.lower()
data_frame['Cliente'] = data_frame['Cliente'].str.replace('[^a-zA-ZáàãâéêíóôõúüçÁÀÃÂÉÊÍÓÔÕÚÜÇ\s]', ' ', regex=True)
data_frame['Cliente'] = data_frame['Cliente'].str.strip()

data_frame['Cliente']

<>:3: SyntaxWarning: invalid escape sequence '\s'
<>:3: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_16673/3489116838.py:3: SyntaxWarning: invalid escape sequence '\s'
  data_frame['Cliente'] = data_frame['Cliente'].str.replace('[^a-zA-ZáàãâéêíóôõúüçÁÀÃÂÉÊÍÓÔÕÚÜÇ\s]', ' ', regex=True)


0         ana  lucia
1     diego armandiu
2     diego armandiu
3     diego armandiu
4     isabely joanes
5     isabely joanes
6     isabely joanes
7     isabely joanes
8     isabely joanes
9       joão gabriel
10    julya meireles
11    julya meireles
12    julya meireles
13       maria julia
14       maria julia
15       maria julia
16       maria julia
17       pedro pasco
18      paulo castro
19     thiago fritzz
Name: Cliente, dtype: str

In [40]:
data_frame.head(20)

,Data de venda,Cliente,Valor da compra
0,2022-06-06,ana lucia,836.50
1,2022-06-06,diego armandiu,573.33
2,2022-06-06,diego armandiu,392.80
3,2022-06-06,diego armandiu,512.34
4,2022-06-07,isabely joanes,825.31
5,2022-06-07,isabely joanes,168.07
6,2022-06-07,isabely joanes,339.18
7,2022-06-07,isabely joanes,314.69
8,2022-06-08,isabely joanes,682.05
9,2022-06-08,joão gabriel,386.34


In [ ]:
# Modificando a formatação das datas da vendas na coluna 'Data de venda':
data_frame['Data de venda'] = pd.to_datetime(data_frame['Data de venda'], format='%d/%m/%Y')

In [43]:
# Deixando as datas somente como o mês e o dia da venda, já que nosso foco está nas vendas da semana:
data_frame['Data de venda'] = data_frame['Data de venda'].dt.strftime('%m-%d')

In [50]:
# Analisando o valor das compras dos clientes durante essa semana (06-06 to 06-10):
data_frame.groupby('Cliente')['Valor da compra'].sum().round(2).sort_values(ascending=False)

Cliente
isabely joanes    2329.30
maria julia       2086.65
julya meireles    1643.74
diego armandiu    1478.47
paulo castro       899.16
thiago fritzz      885.24
ana  lucia         836.50
joão gabriel       386.34
pedro pasco        311.15
Name: Valor da compra, dtype: float64